# B1/B2 Kaggle preparation
Private input bundle, language-explicit CLI17, fixed Llama 3 8B / MiniLM identities.
No download/install/login/BigQuery cells. Prepare the Python 3.11 + exact locked dependencies and genuine local model caches separately.
Finalized EN and VI use separate typed consumers. Select LANGUAGE explicitly; each config schedules three processes per baseline for that language.
Run data preflight first. Native inference is disabled by default; execution remains a separately authorized external run.


In [ ]:
BUNDLE_ARCHIVE = "/kaggle/input/<private-dataset>/input-bundle.tar.gz"
BUNDLE_SHA256 = None  # copy the independently recorded archive hash from handoff
WORKSPACE = "/kaggle/working/b12-input"
OUTPUT_ROOT = "/kaggle/working/b12-runs"
CAMPAIGN_ID = None
MODEL_REVISION = None
ENCODER_REVISION = None
HF_HOME = None  # existing local HF cache root, containing hub/models--.../snapshots/<revision>
ALLOW_REAL_INFERENCE = False
RUN_TIMEOUT_SECONDS = None  # explicit per-process wall-time cap
LANGUAGE = "en"  # choose "en" or "vi"; repeat with separate config/output identity


Set the private dataset archive path and recorded SHA-256. This cell extracts into a fresh workspace and validates the actual data consumers without an encoder or model.


In [ ]:
import hashlib
import os
from pathlib import Path, PurePosixPath
import subprocess
import sys
import tarfile

archive = Path(BUNDLE_ARCHIVE)
if not isinstance(BUNDLE_SHA256, str) or len(BUNDLE_SHA256) != 64:
    raise ValueError("Set the recorded bundle SHA-256 before unpacking")
if hashlib.sha256(archive.read_bytes()).hexdigest() != BUNDLE_SHA256:
    raise ValueError("Bundle SHA-256 mismatch")
workspace = Path(WORKSPACE)
if workspace.exists():
    raise ValueError("Choose a fresh WORKSPACE; existing inputs are never overwritten")
with tarfile.open(archive) as bundle:
    members = bundle.getmembers()
    names = [m.name for m in members]
    if len(names) != len(set(names)) or sum(m.size for m in members) > 128*1024*1024:
        raise ValueError("Invalid archive size/duplicates")
    for member in members:
        name = PurePosixPath(member.name)
        if not member.isfile() or name.is_absolute() or any(p in {".", ".."} or p.startswith(".") for p in member.name.split("/")):
            raise ValueError("Unsafe archive entry")
    bundle.extractall(workspace, filter="data")
allowed_environment = ("PATH", "LD_LIBRARY_PATH", "CUDA_VISIBLE_DEVICES", "LANG", "LC_ALL")
env = {name: os.environ[name] for name in allowed_environment if name in os.environ}
env.update(PYTHONPATH=str(workspace / "src"), PYTHONDONTWRITEBYTECODE="1", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1", HF_HUB_DISABLE_TELEMETRY="1")
entry = [sys.executable, str(workspace / "scripts/25_kaggle_b12.py")]
subprocess.run([*entry, "preflight", "--root", str(workspace), "--level", "data"], cwd=workspace, env=env, check=True)


Supply genuine model/tokenizer revision (shared by CLI17), encoder revision, campaign ID and an existing HF cache. Missing pins/cache/dependencies remain blocked; no placeholders establish readiness.


In [ ]:
import json
config = json.loads((workspace / "configs/kaggle-b12-run.template.json").read_bytes())
config.update(language=LANGUAGE, campaign_id=CAMPAIGN_ID, model_revision=MODEL_REVISION, encoder_revision=ENCODER_REVISION, hf_home=HF_HOME, run_timeout_seconds=RUN_TIMEOUT_SECONDS)
config_path = Path(OUTPUT_ROOT).parent / f"b12-{LANGUAGE}-run-config.json"
payload = json.dumps(config, sort_keys=True, indent=2) + "\n"
if config_path.exists() and config_path.read_text() != payload:
    raise ValueError("Existing config differs; preserve it and choose a new config path")
if not config_path.exists():
    config_path.write_text(payload)
# This checks packages/local snapshot files only; it cannot load a model or GPU.
result = subprocess.run([*entry, "preflight", "--root", str(workspace), "--level", "runtime", "--config", str(config_path), "--output", OUTPUT_ROOT], cwd=workspace, env=env, check=False)
print("Runtime preflight exit:", result.returncode)


Only after runtime preflight succeeds and you authorize GPU inference: enable the setting and run. Each native run retains its own outputs; stop on error, preserve logs, do not tune on benchmark outcomes. Hardware and model compatibility have not been tested by this offline preparation.


In [ ]:
if not ALLOW_REAL_INFERENCE:
    print("BLOCKED: preparation only. Supply genuine pins/caches and opt in separately.")
else:
    subprocess.run([*entry, "run", "--root", str(workspace), "--config", str(config_path), "--output", OUTPUT_ROOT, "--real-inference"], cwd=workspace, env=env, check=True)


Keep all per-run predictions/native logs/native reports/stdout/stderr/run.json and notebook URL/version. Native inference does not establish SQL execution accuracy; T5.4 execution/comparison/privacy/reproducibility remain separate evidence gates.
